# Faster R-CNN seeds on Google Colab

Trains the primary model (Faster R-CNN, ResNet-50 FPN v2) for the remaining seeds with exactly the
settings of seed 0 (`--batch 1`), then scores each on the DENTEX test split.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > **Run all**.

- Checkpoints and results are saved to Google Drive (`MyDrive/caries_runs/`), so if Colab disconnects,
  just **Run all** again: finished seeds are skipped and an interrupted one resumes from its last epoch.
- Keep this tab open while it trains (Colab disconnects idle sessions).
- When both seeds are done, download `MyDrive/caries_runs/frcnn_results.zip` (small, no weights) and put
  it in `~/Downloads` on the Mac.

In [ ]:
SEEDS = [1, 2]  # seeds to train here

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout
assert gpu, "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
print("GPU:", gpu)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!test -d caries || git clone -q https://github.com/felix-chima/explainable-caries-detection.git caries
%cd /content/caries
!git pull -q && git log --oneline -1

In [ ]:
# DENTEX from Hugging Face (fast inside Google), then the same 70/15/15 split as on the Mac
import shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

if not Path("data/dentex_yolo/images/test").exists():
    raw = Path("data/dentex"); raw.mkdir(parents=True, exist_ok=True)
    need = {"training_data.zip", "validation_data.zip", "validation_triple.json"}
    files = [f for f in HfApi().list_repo_files("ibrahimhamamci/DENTEX", repo_type="dataset") if Path(f).name in need]
    assert len(files) == 3, files
    for f in files:
        print("downloading", f)
        shutil.move(hf_hub_download("ibrahimhamamci/DENTEX", f, repo_type="dataset", local_dir="/content/hf"),
                    raw / Path(f).name)
    out = subprocess.run([sys.executable, "scripts/prepare_dentex.py"], capture_output=True, text=True)
    print(out.stdout[-1500:], out.stderr[-1500:])
    assert "split matches" in out.stdout, "Split differs from the Mac's: do not train, report this"
    for z in raw.glob("*.zip"):
        z.unlink()  # free disk
print("dataset ready:", {s: len(list(Path(f"data/dentex_yolo/images/{s}").glob("*.png"))) for s in ("train", "val", "test")})

In [ ]:
# runs/frcnn and runs/eval live on Google Drive so they survive disconnects
DRIVE = Path("/content/drive/MyDrive/caries_runs")
for sub in ("frcnn", "eval"):
    local, remote = Path("runs") / sub, DRIVE / sub
    remote.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        continue
    if local.exists():  # seed-0 files from git: copy across without overwriting
        for f in local.rglob("*"):
            dst = remote / f.relative_to(local)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
        shutil.rmtree(local)
    local.parent.mkdir(exist_ok=True)
    local.symlink_to(remote, target_is_directory=True)
print("runs on Drive:", DRIVE)

In [ ]:
def run(cmd, log):
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line); f.flush()
            if "arning" not in line:
                print(line, end="", flush=True)
        return p.wait()

for seed in SEEDS:
    name = f"frcnn_s{seed}"
    log = Path(f"runs/frcnn/{name}.log")
    if log.exists() and any(l.startswith("done") for l in log.read_text().splitlines()):
        print(name, "already trained")
    else:
        resume = Path(f"runs/frcnn/{name}/weights/last.pt").exists()
        print(name, "resuming" if resume else "starting")
        cmd = [sys.executable, "scripts/train_frcnn.py", "--name", name, "--seed", str(seed), "--batch", "1"]
        assert run(cmd + (["--resume"] if resume else []), log) == 0, f"{name} failed, see {log}"
    if not Path(f"runs/eval/{name}_test.json").exists():
        print("evaluating", name)
        assert run([sys.executable, "scripts/evaluate.py", "frcnn", f"runs/frcnn/{name}/weights/best.pt",
                    "--name", name], log) == 0
    print(open(f"runs/eval/{name}_test.json").read())

In [ ]:
# small results bundle for the Mac (no weights)
!cd runs && rm -f {DRIVE}/frcnn_results.zip && zip -qr {DRIVE}/frcnn_results.zip frcnn eval -x "frcnn/*/weights/*" && ls -la {DRIVE}/frcnn_results.zip
print("Download MyDrive/caries_runs/frcnn_results.zip and put it in ~/Downloads on the Mac.")